<a href="https://colab.research.google.com/github/dhanushkumar2968/Car-colour-detection-Model/blob/main/sign_language_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🤟 Sign Language Detection System

**What this project does:**
- Trains a model to recognize **ASL (American Sign Language)** hand signs A–Z
- Works only between **6 PM – 10 PM** (time-lock feature)
- GUI with **Upload Image** mode
- GUI with **Real-Time Webcam** mode

---
**How it works (simple explanation):**
1. We download 87,000 hand sign images from Kaggle
2. We train a small neural network (CNN) to recognize each letter
3. We use the trained model to predict signs from your image or webcam

---

### ✅ CELL 1 — Install Required Libraries

In [1]:
# Install everything we need
!pip install tensorflow opencv-python-headless mediapipe ipywidgets Pillow --quiet
!pip install kaggle --quiet

print("✅ All libraries installed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 43.3 MB/s eta 0:00:00
✅ All libraries installed!


### ✅ CELL 2 — Connect Kaggle API

In [2]:
import os, json

# ✏️ PASTE YOUR KAGGLE USERNAME AND KEY HERE
KAGGLE_USERNAME = "dhanush2968"   # ← change this
KAGGLE_KEY      = "colab_new"        # ← change this

# Save kaggle credentials automatically
os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump({"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)

print("✅ Kaggle API connected!")

✅ Kaggle API connected!


### ✅ CELL 3 — Download ASL Dataset from Kaggle

In [3]:
# Download ASL Alphabet dataset (87,000 images, A-Z + space + delete + nothing)
!kaggle datasets download -d grassknoted/asl-alphabet --unzip -p /content/asl_data/

# Check what was downloaded
import os
train_path = '/content/asl_data/asl_alphabet_train/asl_alphabet_train'
folders = os.listdir(train_path)
print(f"✅ Dataset downloaded!")
print(f"📁 Found {len(folders)} sign categories: {sorted(folders)}")

Dataset URL: https://www.kaggle.com/datasets/grassknoted/asl-alphabet
License(s): GPL-2.0
100% 1.03G/1.03G [00:37<00:00, 29.3MB/s]

✅ Dataset downloaded!
📁 Found 29 sign categories: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'del', 'nothing', 'space']


### ✅ CELL 4 — Import Libraries & Time-Lock Setup

In [5]:
import numpy as np
import os
import cv2
from PIL import Image
import matplotlib.pyplot as plt
from datetime import datetime
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import warnings
warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────────────
# TIME-LOCK: Model only works between 6 PM and 10 PM
# ─────────────────────────────────────────────────────
def is_within_allowed_time():
    """
    Returns True only if current time is between 6:00 PM and 10:00 PM.
    You can change the hours below.
    """
    now = datetime.now()
    current_hour = now.hour  # 0-23 format (18 = 6PM, 22 = 10PM)

    START_HOUR = 18   # 6:00 PM  ← change this if needed
    END_HOUR   = 22   # 10:00 PM ← change this if needed

    if START_HOUR <= current_hour < END_HOUR:
        return True
    else:
        return False

# ─────────────────────────────────────────────────────
# Quick test of the time check
now = datetime.now()
print(f"⏰ Current time: {now.strftime('%I:%M %p')}")
if is_within_allowed_time():
    print("✅ System is ACTIVE (within 6 PM – 10 PM)")
else:
    print("🔒 System is LOCKED (only works 6 PM – 10 PM)")
    print("   ℹ️  For testing, you can remove this check in the predict function")

print("\n✅ Libraries loaded!")

⏰ Current time: 02:01 AM
🔒 System is LOCKED (only works 6 PM – 10 PM)
   ℹ️  For testing, you can remove this check in the predict function

✅ Libraries loaded!


### ✅ CELL 5 — Prepare Training Data

In [6]:
# ─────────────────────────────────────────────────────
# SETTINGS — change these if needed
# ─────────────────────────────────────────────────────
IMG_SIZE     = 64      # Each image resized to 64x64 pixels
BATCH_SIZE   = 32      # How many images to process at once
EPOCHS       = 10      # How many times to train on the full dataset

# We only use letters A-Z (26 classes) — skip space/delete/nothing
SELECTED_CLASSES = [chr(i) for i in range(ord('A'), ord('Z')+1)]
print(f"📚 Training on {len(SELECTED_CLASSES)} sign classes: {SELECTED_CLASSES}")

# ─────────────────────────────────────────────────────
# Load images using Keras ImageDataGenerator
# This automatically reads folders and labels images
# ─────────────────────────────────────────────────────
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Normalize pixel values from 0-255 to 0-1
# Also split 20% for validation automatically
datagen = ImageDataGenerator(
    rescale=1./255,          # Normalize pixel values
    validation_split=0.2,    # 80% train, 20% validation
    horizontal_flip=True,    # Randomly flip images (data augmentation)
    zoom_range=0.1           # Slight zoom (data augmentation)
)

train_path = '/content/asl_data/asl_alphabet_train/asl_alphabet_train'

# Training data
train_data = datagen.flow_from_directory(
    train_path,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    subset='training',
    classes=SELECTED_CLASSES,
    class_mode='categorical'
)

# Validation data
val_data = datagen.flow_from_directory(
    train_path,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    subset='validation',
    classes=SELECTED_CLASSES,
    class_mode='categorical'
)

# Save class labels (A, B, C... Z)
CLASS_NAMES = list(train_data.class_indices.keys())
print(f"\n✅ Data ready! Classes: {CLASS_NAMES}")
print(f"   Training images  : {train_data.samples}")
print(f"   Validation images: {val_data.samples}")

📚 Training on 26 sign classes: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z']
Found 62400 images belonging to 26 classes.
Found 15600 images belonging to 26 classes.

✅ Data ready! Classes: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z']
   Training images  : 62400
   Validation images: 15600


### ✅ CELL 6 — Build & Train the CNN Model

In [7]:
# ─────────────────────────────────────────────────────
# BUILD THE MODEL
# CNN = Convolutional Neural Network
# Think of it as: image → find edges → find shapes → find patterns → predict letter
# ─────────────────────────────────────────────────────

NUM_CLASSES = len(CLASS_NAMES)  # 26 letters

model = keras.Sequential([

    # ── Input Layer ──
    layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),  # 64x64 RGB image

    # ── Block 1: Detect basic shapes ──
    layers.Conv2D(32, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2, 2),      # Shrink image by half
    layers.Dropout(0.25),           # Prevent overfitting

    # ── Block 2: Detect complex patterns ──
    layers.Conv2D(64, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2, 2),
    layers.Dropout(0.25),

    # ── Block 3: Detect hand shapes ──
    layers.Conv2D(128, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2, 2),
    layers.Dropout(0.25),

    # ── Flatten and Classify ──
    layers.Flatten(),               # Convert 2D to 1D
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(NUM_CLASSES, activation='softmax')  # Output: probability for each letter
])

# Compile the model (choose optimizer and loss function)
model.compile(
    optimizer='adam',                     # Adam optimizer (works well for most tasks)
    loss='categorical_crossentropy',      # Loss for multi-class classification
    metrics=['accuracy']                  # We want to track accuracy
)

model.summary()
print("\n✅ Model built!")

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 64, 64, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 64, 64, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 32, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 32, 32, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 32, 32, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 16, 16, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 16, 16, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 8, 8, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 8, 8, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 8192)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │     2,097,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 26)             │         6,682 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,198,234 (8.39 MB)

 Trainable params: 2,197,786 (8.38 MB)

 Non-trainable params: 448 (1.75 KB)


✅ Model built!


In [8]:
# ─────────────────────────────────────────────────────
# TRAIN THE MODEL
# This will take 15-30 minutes on Colab GPU
# Tip: Runtime → Change runtime type → GPU  (makes it 10x faster!)
# ─────────────────────────────────────────────────────

from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

callbacks = [
    # Save the best model automatically
    ModelCheckpoint('best_model.h5', save_best_only=True, monitor='val_accuracy', verbose=1),

    # Stop training if no improvement after 5 epochs (saves time)
    EarlyStopping(patience=5, monitor='val_accuracy', restore_best_weights=True),

    # Reduce learning rate if stuck
    ReduceLROnPlateau(patience=3, factor=0.5, verbose=1)
]

print("🚀 Starting training... (this may take 15-30 minutes)")
print("💡 Tip: Go to Runtime → Change runtime type → GPU for faster training!\n")

history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=EPOCHS,
    callbacks=callbacks
)

print("\n✅ Training complete! Model saved as 'best_model.h5'")

🚀 Starting training... (this may take 15-30 minutes)
💡 Tip: Go to Runtime → Change runtime type → GPU for faster training!

Epoch 1/10
  67/1950 ━━━━━━━━━━━━━━━━━━━━ 14:20 457ms/step - accuracy: 0.0538 - loss: 5.0683

KeyboardInterrupt: 

In [ ]:
# ─────────────────────────────────────────────────────
# PLOT TRAINING RESULTS
# Shows how accuracy improved over each epoch
# ─────────────────────────────────────────────────────

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy graph
ax1.plot(history.history['accuracy'],     label='Train Accuracy',      color='blue',   linewidth=2)
ax1.plot(history.history['val_accuracy'], label='Validation Accuracy', color='orange', linewidth=2)
ax1.set_title('Model Accuracy', fontsize=14, fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Loss graph
ax2.plot(history.history['loss'],     label='Train Loss',      color='blue',   linewidth=2)
ax2.plot(history.history['val_loss'], label='Validation Loss', color='orange', linewidth=2)
ax2.set_title('Model Loss', fontsize=14, fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_graph.png', dpi=150)
plt.show()
print("✅ Training graph saved!")

### ✅ CELL 7 — Prediction Function

In [ ]:
import numpy as np
from PIL import Image

# Load the best saved model
model = keras.models.load_model('best_model.h5')
print("✅ Best model loaded!")

# ─────────────────────────────────────────────────────
# PREDICT FUNCTION
# Takes an image → returns predicted letter + confidence
# ─────────────────────────────────────────────────────
def predict_sign(image_input, bypass_time_check=False):
    """
    Predicts the sign language letter from an image.

    Parameters:
        image_input: file path (string) OR numpy array (from webcam)
        bypass_time_check: set True to test outside 6-10 PM

    Returns:
        predicted_letter (str), confidence (float), annotated_image (numpy array)
    """

    # ── TIME CHECK ──
    if not bypass_time_check and not is_within_allowed_time():
        now = datetime.now()
        msg = f"🔒 System locked! Current time: {now.strftime('%I:%M %p')}. Works only 6 PM - 10 PM."
        print(msg)
        return None, 0, None

    # ── LOAD IMAGE ──
    if isinstance(image_input, str):
        # If file path is given
        img = Image.open(image_input).convert('RGB')
        img_array = np.array(img)
    else:
        # If numpy array is given (from webcam)
        img_array = image_input.copy()
        if img_array.shape[2] == 3:
            img = Image.fromarray(img_array)

    # ── PREPROCESS: Resize and normalize ──
    img_resized = img.resize((IMG_SIZE, IMG_SIZE))
    img_normalized = np.array(img_resized) / 255.0   # Scale 0-1
    img_batch = np.expand_dims(img_normalized, axis=0)  # Add batch dimension

    # ── PREDICT ──
    predictions = model.predict(img_batch, verbose=0)
    predicted_index = np.argmax(predictions[0])         # Index of highest probability
    predicted_letter = CLASS_NAMES[predicted_index]     # Get letter name
    confidence = predictions[0][predicted_index] * 100  # Convert to percentage

    # Top 3 predictions
    top3_indices = np.argsort(predictions[0])[::-1][:3]
    top3 = [(CLASS_NAMES[i], round(predictions[0][i]*100, 1)) for i in top3_indices]

    # ── DRAW RESULT ON IMAGE ──
    result_img = img_array.copy()
    h, w = result_img.shape[:2]

    # Dark background bar at bottom
    overlay = result_img.copy()
    cv2.rectangle(overlay, (0, h-90), (w, h), (0, 0, 0), -1)
    result_img = cv2.addWeighted(overlay, 0.7, result_img, 0.3, 0)

    # Predicted letter (large)
    cv2.putText(result_img, f'{predicted_letter}',
                (15, h-15), cv2.FONT_HERSHEY_SIMPLEX, 2.5, (0,255,0), 4)

    # Confidence percentage
    cv2.putText(result_img, f'{confidence:.1f}%',
                (90, h-15), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255,255,0), 2)

    # Top 3 labels at top
    for i, (letter, conf) in enumerate(top3):
        cv2.putText(result_img, f'#{i+1} {letter}: {conf}%',
                    (10, 28 + i*28), cv2.FONT_HERSHEY_SIMPLEX, 0.7,
                    (255,255,255), 2)

    return predicted_letter, confidence, result_img

print("✅ Prediction function ready!")

### ✅ CELL 8 — GUI: Upload Image Mode

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
from google.colab import files

# ─────────────────────────────────────────────────────
# GUI LAYOUT
# ─────────────────────────────────────────────────────

title = widgets.HTML("""
    <div style='background:#1a1a2e; padding:15px; border-radius:10px; text-align:center;'>
        <h2 style='color:#e94560; margin:0;'>🤟 Sign Language Detection</h2>
        <p style='color:#aaa; margin:5px 0 0;'>Upload a hand sign image to detect the letter</p>
    </div>
""")

upload_btn = widgets.Button(
    description='📁 Upload Hand Sign Image',
    button_style='primary',
    layout=widgets.Layout(width='250px', height='45px', margin='10px 0')
)

# Toggle for testing outside 6-10 PM
bypass_toggle = widgets.Checkbox(
    value=False,
    description='🔓 Bypass time restriction (for testing)',
    style={'description_width': 'initial'}
)

output_area = widgets.Output()

def on_upload(b):
    with output_area:
        clear_output(wait=True)

        # Time check
        if not bypass_toggle.value and not is_within_allowed_time():
            now = datetime.now()
            print(f"🔒 System locked! Time: {now.strftime('%I:%M %p')}")
            print("   Works only 6:00 PM – 10:00 PM")
            print("   ✅ Tick the bypass checkbox above to test anytime.")
            return

        print("📂 Select a hand sign image (JPG or PNG)...")
        uploaded = files.upload()

        for filename, content in uploaded.items():
            # Save uploaded file
            save_path = f'/content/{filename}'
            with open(save_path, 'wb') as f:
                f.write(content)

            print(f"\n🔍 Analysing: {filename}...")

            # Predict
            letter, conf, result_img = predict_sign(
                save_path, bypass_time_check=bypass_toggle.value
            )

            if result_img is not None:
                # Show side-by-side
                original = np.array(Image.open(save_path).convert('RGB'))

                fig, axes = plt.subplots(1, 2, figsize=(12, 5))
                fig.patch.set_facecolor('#1a1a2e')

                axes[0].imshow(original)
                axes[0].set_title('📷 Original Image', color='white', fontsize=13, pad=10)
                axes[0].axis('off')
                axes[0].set_facecolor('#1a1a2e')

                axes[1].imshow(result_img)
                axes[1].set_title(f'🤟 Predicted: {letter}  ({conf:.1f}%)',
                                  color='#00ff88', fontsize=13, pad=10, fontweight='bold')
                axes[1].axis('off')
                axes[1].set_facecolor('#1a1a2e')

                plt.tight_layout()
                plt.savefig('/content/sign_result.jpg', dpi=150,
                            bbox_inches='tight', facecolor='#1a1a2e')
                plt.show()

                print(f"\n✅ Prediction: '{letter}' with {conf:.1f}% confidence")
                print("📥 Result saved as /content/sign_result.jpg")

upload_btn.on_click(on_upload)

# Show the GUI
display(title, bypass_toggle, upload_btn, output_area)

### ✅ CELL 9 — GUI: Real-Time Webcam Mode

In [ ]:
# ─────────────────────────────────────────────────────
# REAL-TIME WEBCAM DETECTION
# Uses JavaScript to capture your webcam in the browser
# and sends each frame to Python for prediction
# ─────────────────────────────────────────────────────

from IPython.display import Javascript, display
from google.colab.output import eval_js
from base64 import b64decode, b64encode
import numpy as np
import cv2
import ipywidgets as widgets

# ── JavaScript code to open webcam and capture a single frame ──
WEBCAM_JS = """
async function captureFrame() {
    // Create video element
    const video = document.createElement('video');
    const stream = await navigator.mediaDevices.getUserMedia({video: true});
    video.srcObject = stream;
    await video.play();

    // Short delay so camera warms up
    await new Promise(r => setTimeout(r, 1000));

    // Draw frame to canvas
    const canvas = document.createElement('canvas');
    canvas.width  = video.videoWidth  || 640;
    canvas.height = video.videoHeight || 480;
    canvas.getContext('2d').drawImage(video, 0, 0);

    // Stop webcam
    stream.getTracks().forEach(track => track.stop());

    // Return as base64 JPEG
    return canvas.toDataURL('image/jpeg', 0.8);
}
captureFrame();
"""

def capture_webcam_frame():
    """Captures a single frame from webcam. Returns numpy array."""
    try:
        data_url = eval_js(WEBCAM_JS)
        # Remove the 'data:image/jpeg;base64,' prefix
        img_bytes = b64decode(data_url.split(',')[1])
        img_array = np.frombuffer(img_bytes, dtype=np.uint8)
        frame = cv2.imdecode(img_array, cv2.IMREAD_COLOR)
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        return frame_rgb
    except Exception as e:
        print(f"❌ Webcam error: {e}")
        return None


# ── GUI for Webcam Mode ──
webcam_title = widgets.HTML("""
    <div style='background:#0f3460; padding:15px; border-radius:10px; text-align:center;'>
        <h2 style='color:#e94560; margin:0;'>📷 Real-Time Webcam Detection</h2>
        <p style='color:#aaa; margin:5px 0 0;'>Show a hand sign to your webcam → click Capture</p>
    </div>
""")

capture_btn = widgets.Button(
    description='📸 Capture & Detect',
    button_style='success',
    layout=widgets.Layout(width='200px', height='45px', margin='10px 0')
)

bypass_webcam = widgets.Checkbox(
    value=False,
    description='🔓 Bypass time restriction (for testing)',
    style={'description_width': 'initial'}
)

webcam_output = widgets.Output()

def on_capture(b):
    with webcam_output:
        clear_output(wait=True)

        # Time check
        if not bypass_webcam.value and not is_within_allowed_time():
            now = datetime.now()
            print(f"🔒 Locked! Time: {now.strftime('%I:%M %p')} | Works 6 PM – 10 PM only")
            print("   ✅ Tick the bypass checkbox to test anytime.")
            return

        print("📷 Opening webcam... (allow camera access when asked)")
        frame = capture_webcam_frame()

        if frame is not None:
            print("🔍 Analysing hand sign...")

            # Predict directly from numpy array
            letter, conf, result_img = predict_sign(
                frame, bypass_time_check=bypass_webcam.value
            )

            if result_img is not None:
                fig, axes = plt.subplots(1, 2, figsize=(12, 5))
                fig.patch.set_facecolor('#0f3460')

                axes[0].imshow(frame)
                axes[0].set_title('📷 Captured Frame', color='white', fontsize=13)
                axes[0].axis('off')

                axes[1].imshow(result_img)
                axes[1].set_title(f'🤟 Predicted: {letter}  ({conf:.1f}%)',
                                  color='#00ff88', fontsize=13, fontweight='bold')
                axes[1].axis('off')

                plt.tight_layout()
                plt.savefig('/content/webcam_result.jpg', dpi=150,
                            bbox_inches='tight', facecolor='#0f3460')
                plt.show()

                print(f"\n✅ Sign detected: '{letter}' with {conf:.1f}% confidence")
                print("💡 Hold sign clearly in good lighting for better accuracy")
        else:
            print("❌ Could not capture frame. Try again or use Upload mode.")

capture_btn.on_click(on_capture)

# Show the webcam GUI
display(webcam_title, bypass_webcam, capture_btn, webcam_output)

---
### ✅ CELL 10 — Quick Test (No Webcam / No Upload Needed)
Test with a sample image from the dataset to verify everything works.

In [ ]:
import glob

# Pick any sample image from the dataset
sample_images = glob.glob('/content/asl_data/asl_alphabet_train/asl_alphabet_train/**/*.jpg',
                          recursive=True)

print(f"Testing on 6 sample images from the dataset...\n")

fig, axes = plt.subplots(2, 3, figsize=(14, 9))
fig.patch.set_facecolor('#1a1a2e')
axes = axes.flatten()

# Pick one image per letter (first 6 letters A-F)
test_letters = ['A', 'B', 'C', 'D', 'E', 'F']

for i, letter in enumerate(test_letters):
    # Find an image of this letter
    letter_images = glob.glob(
        f'/content/asl_data/asl_alphabet_train/asl_alphabet_train/{letter}/*.jpg'
    )

    if not letter_images:
        continue

    test_img = letter_images[0]

    # Predict (bypass time check for testing)
    pred_letter, conf, result_img = predict_sign(test_img, bypass_time_check=True)

    if result_img is not None:
        correct = "✅" if pred_letter == letter else "❌"
        axes[i].imshow(result_img)
        axes[i].set_title(f'{correct} True: {letter} → Pred: {pred_letter} ({conf:.0f}%)',
                          color='#00ff88' if pred_letter == letter else '#ff4444',
                          fontsize=10)
        axes[i].axis('off')

plt.suptitle('Quick Test Results', color='white', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('/content/test_results.jpg', dpi=150, bbox_inches='tight', facecolor='#1a1a2e')
plt.show()
print("✅ Quick test complete!")